# Label Aggregation — raw_crowd_test.tsv

Aggregates `annotator_emo` by `hash_id` using:
- **Majority Vote** — простое голосование большинством
- **Dawid-Skene** с порогами уверенности: 0.85, 0.90, 0.95, 0.98

Записи с уверенностью ниже порога исключаются из выходного файла.

**Вход:** `raw_crowd_test.tsv`  
**Выход:** `aggregated/aggregated_majority.tsv`, `aggregated/aggregated_ds_0.85.tsv`, ...

## 1. Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'crowd-kit'], check=True)
print('Done.')

## 2. Load data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_TSV  = 'raw_crowd_test.tsv'   # путь к входному файлу
OUTPUT_DIR = Path('aggregated')
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_TSV, sep='\t')
print(f'Rows: {len(df):,}  |  unique items: {df["hash_id"].nunique():,}  |  unique annotators: {df["annotator_id"].nunique():,}')
print('Labels:', sorted(df['annotator_emo'].unique()))
df.head(3)

## 3. Prepare crowd-kit format

In [ ]:
# Метаданные — одна строка на hash_id
META_COLS = ['hash_id', 'audio_path', 'duration', 'golden_emo',
             'speaker_text', 'speaker_emo', 'source_id']
meta = df[META_COLS].drop_duplicates('hash_id').set_index('hash_id')

# crowd-kit ожидает колонки: task / worker / label
crowd = (
    df[['hash_id', 'annotator_id', 'annotator_emo']]
    .rename(columns={'hash_id': 'task',
                     'annotator_id': 'worker',
                     'annotator_emo': 'label'})
)
print(f'crowd shape: {crowd.shape}')
crowd.head(3)

## 4. Helper: save result

In [ ]:
def save_result(agg_labels: pd.Series,
                confidence: pd.Series | None,
                threshold: float | None,
                tag: str) -> pd.DataFrame:
    result = agg_labels.rename('aggregated_emo').to_frame()

    if confidence is not None:
        result['confidence'] = confidence
        if threshold is not None:
            before = len(result)
            result = result[result['confidence'] >= threshold]
            kept_pct = 100 * len(result) / before
            print(f'  threshold={threshold}: kept {len(result):,} / {before:,} ({kept_pct:.1f}%)')

    out = (
        result
        .join(meta, how='left')
        .reset_index()
        .rename(columns={'task': 'hash_id'})
    )
    path = OUTPUT_DIR / f'aggregated_{tag}.tsv'
    out.to_csv(path, sep='\t', index=False)
    print(f'  Saved → {path}  ({len(out):,} rows)')
    return out

## 5. Majority Vote

In [ ]:
from crowdkit.aggregation import MajorityVote

mv_labels = MajorityVote().fit_predict(crowd)

print('Majority Vote — распределение меток:')
print(mv_labels.value_counts().to_string())
print()
mv_out = save_result(mv_labels, confidence=None, threshold=None, tag='majority')

## 6. Dawid-Skene

In [ ]:
from crowdkit.aggregation import DawidSkene

THRESHOLDS = [0.85, 0.90, 0.95, 0.98]

print('Fitting Dawid-Skene (100 iterations)...')
ds = DawidSkene(n_iter=100)
ds.fit(crowd)

# probas_: DataFrame shape (n_tasks, n_labels), строки суммируются в 1
probs     = ds.probas_
ds_labels = probs.idxmax(axis=1)   # наиболее вероятная метка
ds_conf   = probs.max(axis=1)      # уверенность = максимальная апостериорная вероятность

print(f'\nDawid-Skene — все {len(ds_labels):,} items:')
print(ds_labels.value_counts().to_string())
print()

for t in THRESHOLDS:
    print(f'--- threshold={t} ---')
    save_result(ds_labels, ds_conf, threshold=t, tag=f'ds_{t}')
    print()

## 7. Summary + confidence distribution

In [ ]:
import matplotlib.pyplot as plt

# таблица с количеством записей в каждом файле
rows = []
for path in sorted(OUTPUT_DIR.glob('aggregated_*.tsv')):
    d = pd.read_csv(path, sep='\t')
    rows.append({'file': path.name, 'items': len(d)})
print(pd.DataFrame(rows).to_string(index=False))

# гистограмма уверенности Dawid-Skene
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ds_conf.values, bins=60, edgecolor='white', linewidth=0.4, color='steelblue')
for t in THRESHOLDS:
    ax.axvline(t, linestyle='--', linewidth=1.2, label=str(t))
ax.set_xlabel('Confidence (max posterior probability)')
ax.set_ylabel('Items')
ax.set_title('Dawid-Skene confidence distribution')
ax.legend(title='threshold')
plt.tight_layout()
plt.show()